# H3 Dedicated-Hard training and saved validation evidence

Reviewer-safe: training is disabled by default. This notebook never evaluates the frozen test. Missing VM artifacts remain missing. The approved `block_a_two_task_v1` extension defines four-class endpoint validation macro-F1 as the checkpoint-selection metric for H2/H3/H4.

In [ ]:
SYSTEM_TYPE = "dedicated_hard"
BACKBONE = "efficientnet_b0"
SEED = 42
RUN_TRAINING = False
RESUME = False

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.experiment_catalog import experiment_config
from src.results_registry import matrix_status, read_csv
from src.run_experiment import run_experiment
config = experiment_config(SYSTEM_TYPE, BACKBONE, SEED)
run_id = config["experiment_id"]

In [ ]:
status = matrix_status()
display(status.loc[status.run_id == run_id])
if RUN_TRAINING:
    result = run_experiment(SYSTEM_TYPE, BACKBONE, SEED, resume=RESUME, allow_training=True)
    display(result)
else:
    print("Review mode: no training requested. PENDING means this experiment has not been run.")

In [ ]:
from src.reporting import run_status
summary, history = run_status(run_id)
display(summary)
if not history.empty:
    display(history)
rows = [r for r in read_csv(ROOT / "results/master_results.csv") if r["run_id"] == run_id]
if rows:
    import pandas as pd
    display(pd.DataFrame(rows))
else:
    print("No saved results available locally. Run the artifact index rebuild on Azure when artifacts exist.")

Set `SKIN_CANCER_DATA_ROOT` on Azure before explicitly enabling training. Completed experiments are skipped; resume requires matching configuration and checkpoint metadata. The selected checkpoint(s), full validation probabilities, metrics and central indexes are saved by `src/`.